In [0]:
%pip install prophet

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import numpy as np
from prophet import Prophet
import warnings
warnings.filterwarnings("ignore")
 
DB = "hospital_blackhole"
spark.sql(f"USE {DB}")


In [0]:
# GOLD TABLE 1 — gold_dropout_funnel
# All-India + per-state funnel with absolute numbers + rates
# ============================================================
 
# COMMAND ----------
df_j = spark.table(f"{DB}.silver_patient_journeys")
df_dist = spark.table(f"{DB}.bronze_district_master")
 
# All-India aggregate
funnel_india = df_j.agg(
    F.sum("cases_diagnosed_phc").alias("stage0_diagnosed"),
    F.sum("cases_referred").alias("stage1_referred"),
    F.sum("cases_chc_attended").alias("stage2_chc"),
    F.sum("cases_specialist_ref").alias("stage3_specialist"),
    F.sum("cases_tertiary").alias("stage4_tertiary"),
    F.sum("cases_treatment_started").alias("stage5_treatment"),
    F.sum("cases_completed").alias("stage6_completed"),
    F.sum("dropout_stage1").alias("d1"),
    F.sum("dropout_stage2").alias("d2"),
    F.sum("dropout_stage3").alias("d3"),
    F.sum("dropout_stage4").alias("d4"),
    F.sum("dropout_stage5").alias("d5"),
    F.sum("dropout_stage6").alias("d6"),
).withColumn("scope", F.lit("All India")) \
 .withColumn("disease_category", F.lit("All diseases"))
 
# Per disease aggregate
funnel_disease = df_j.groupBy("disease_category").agg(
    F.sum("cases_diagnosed_phc").alias("stage0_diagnosed"),
    F.sum("cases_referred").alias("stage1_referred"),
    F.sum("cases_chc_attended").alias("stage2_chc"),
    F.sum("cases_specialist_ref").alias("stage3_specialist"),
    F.sum("cases_tertiary").alias("stage4_tertiary"),
    F.sum("cases_treatment_started").alias("stage5_treatment"),
    F.sum("cases_completed").alias("stage6_completed"),
    F.sum("dropout_stage1").alias("d1"),
    F.sum("dropout_stage2").alias("d2"),
    F.sum("dropout_stage3").alias("d3"),
    F.sum("dropout_stage4").alias("d4"),
    F.sum("dropout_stage5").alias("d5"),
    F.sum("dropout_stage6").alias("d6"),
).withColumn("scope", F.lit("By Disease"))
 
# Union and compute rates
def add_funnel_rates(df):
    return df \
        .withColumn("rate1_referral",
            F.round(F.col("stage1_referred") / F.greatest(F.col("stage0_diagnosed"), F.lit(1)) * 100, 1)) \
        .withColumn("rate2_chc",
            F.round(F.col("stage2_chc") / F.greatest(F.col("stage1_referred"), F.lit(1)) * 100, 1)) \
        .withColumn("rate3_specialist",
            F.round(F.col("stage3_specialist") / F.greatest(F.col("stage2_chc"), F.lit(1)) * 100, 1)) \
        .withColumn("rate4_tertiary",
            F.round(F.col("stage4_tertiary") / F.greatest(F.col("stage3_specialist"), F.lit(1)) * 100, 1)) \
        .withColumn("rate5_treatment",
            F.round(F.col("stage5_treatment") / F.greatest(F.col("stage4_tertiary"), F.lit(1)) * 100, 1)) \
        .withColumn("rate6_completed",
            F.round(F.col("stage6_completed") / F.greatest(F.col("stage5_treatment"), F.lit(1)) * 100, 1)) \
        .withColumn("overall_completion_pct",
            F.round(F.col("stage6_completed") / F.greatest(F.col("stage0_diagnosed"), F.lit(1)) * 100, 1)) \
        .withColumn("total_dropped",
            F.col("d1")+F.col("d2")+F.col("d3")+F.col("d4")+F.col("d5")+F.col("d6")) \
        .withColumn("refreshed_at", F.current_timestamp())
 
# Align schemas before union
india_cols = funnel_india.columns
disease_cols = funnel_disease.columns
for c in india_cols:
    if c not in disease_cols:
        funnel_disease = funnel_disease.withColumn(c, F.lit(None))
for c in disease_cols:
    if c not in india_cols:
        funnel_india = funnel_india.withColumn(c, F.lit(None))
 
df_funnel = add_funnel_rates(funnel_india.unionByName(funnel_disease, allowMissingColumns=True))
df_funnel.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable(f"{DB}.gold_dropout_funnel")
print(f"✅ gold_dropout_funnel: {df_funnel.count()} rows")
 


In [0]:
# GOLD TABLE 2 — gold_bhs_district
# Black Hole Score per district — the centrepiece Gold table
# BHS = weighted composite of 5 signals, 0–100
# ============================================================
 
# COMMAND ----------
df_dropout = spark.table(f"{DB}.silver_dropout_events")
df_cap     = spark.table(f"{DB}.silver_facility_capacity")
df_geo     = spark.table(f"{DB}.silver_geo_barriers")
df_pmjay   = spark.table(f"{DB}.bronze_pmjay_claims")
df_pred    = spark.table(f"{DB}.silver_dropout_predictions")
df_nfhs    = spark.table(f"{DB}.bronze_nfhs_baseline")
 
# District-level dropout aggregate (all diseases)
dist_dropout = df_dropout.groupBy("district_id","district_name","state").agg(
    F.round(F.avg("overall_dropout_pct"), 1).alias("avg_dropout_pct"),
    F.round(F.avg("stage1_pct"), 1).alias("avg_stage1_pct"),
    F.round(F.avg("stage2_pct"), 1).alias("avg_stage2_pct"),  # biggest dropout
    F.round(F.avg("completion_pct"), 1).alias("avg_completion_pct"),
    F.sum("total_diagnosed").alias("total_diagnosed"),
    F.sum("total_dropped").alias("total_dropped"),
    F.first("vulnerability_idx").alias("vulnerability_idx"),
)
 
# PMJAY claim gap
pmjay_dist = df_pmjay.groupBy("district_id").agg(
    F.round(F.avg("claim_gap_pct"), 1).alias("pmjay_claim_gap_pct"),
    F.round(F.avg("utilisation_rate") * 100, 2).alias("pmjay_utilisation_pct"),
)
 
# Join all signals
df_bhs = dist_dropout \
    .join(df_cap.select("district_id","avg_travel_hrs_to_dh","avg_specialty_availability_pct",
                         "avg_vacancy_rate","n_dh","n_chc"), on="district_id", how="left") \
    .join(df_geo.select("district_id","geo_barrier_score","tribal_pct","population",
                         "lat","lon"), on="district_id", how="left") \
    .join(pmjay_dist, on="district_id", how="left") \
    .join(df_pred.groupBy("district_id").agg(
        F.avg("dropout_prob_xgb").alias("ml_dropout_prob")), on="district_id", how="left") \
    .join(df_nfhs.select("district_id","wasting_under5_pct","anaemia_women_pct"),
          on="district_id", how="left") \
    .fillna(0)
 
# Normalize each component 0–1 (min-max global)
def norm_col(df, col_name, new_name, invert=False):
    mn = df.agg(F.min(col_name)).collect()[0][0] or 0
    mx = df.agg(F.max(col_name)).collect()[0][0] or 1
    rng = max(mx - mn, 0.001)
    if invert:
        return df.withColumn(new_name,
            F.round(1.0 - (F.col(col_name) - mn) / rng, 4))
    else:
        return df.withColumn(new_name,
            F.round((F.col(col_name) - mn) / rng, 4))
 
df_bhs = norm_col(df_bhs, "avg_stage2_pct", "n_referral_gap")        # stage2 = biggest dropout
df_bhs = norm_col(df_bhs, "avg_dropout_pct", "n_dropout_rate")
df_bhs = norm_col(df_bhs, "avg_travel_hrs_to_dh", "n_travel_barrier")
df_bhs = norm_col(df_bhs, "avg_specialty_availability_pct", "n_specialist_deficit", invert=True)
df_bhs = norm_col(df_bhs, "pmjay_claim_gap_pct", "n_pmjay_gap")
 
# BHS Formula (weights from domain literature)
df_bhs = df_bhs.withColumn(
    "bhs_raw",
    F.col("n_referral_gap")        * 0.30 +
    F.col("n_dropout_rate")        * 0.25 +
    F.col("n_travel_barrier")      * 0.20 +
    F.col("n_specialist_deficit")  * 0.15 +
    F.col("n_pmjay_gap")           * 0.10
)
 
# Scale to 0–100
bhs_min = df_bhs.agg(F.min("bhs_raw")).collect()[0][0]
bhs_max = df_bhs.agg(F.max("bhs_raw")).collect()[0][0]
bhs_rng = max(bhs_max - bhs_min, 0.001)
 
df_bhs = df_bhs.withColumn(
    "bhs_score",
    F.round((F.col("bhs_raw") - bhs_min) / bhs_rng * 100, 1)
)
 
# Risk tier
df_bhs = df_bhs.withColumn(
    "risk_tier",
    F.when(F.col("bhs_score") >= 75, "Critical")
     .when(F.col("bhs_score") >= 55, "High Risk")
     .when(F.col("bhs_score") >= 35, "Moderate")
     .otherwise("Low Risk")
).withColumn(
    "city_rank",
    F.row_number().over(Window.partitionBy("state").orderBy(F.col("bhs_score").desc()))
).withColumn(
    "national_rank",
    F.row_number().over(Window.orderBy(F.col("bhs_score").desc()))
).withColumn(
    "patients_at_risk",
    F.round(F.col("total_diagnosed") * F.col("avg_dropout_pct") / 100, 0).cast("long")
).withColumn("refreshed_at", F.current_timestamp())
 
df_bhs.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable(f"{DB}.gold_bhs_district")
spark.sql(f"OPTIMIZE {DB}.gold_bhs_district ZORDER BY (bhs_score)")
print(f"✅ gold_bhs_district: {df_bhs.count()} districts")
print("\nTop 10 worst districts:")
df_bhs.orderBy("bhs_score", ascending=False).select(
    "district_name","state","bhs_score","risk_tier",
    "avg_dropout_pct","avg_stage2_pct","avg_travel_hrs_to_dh"
).show(10)


In [0]:
# ============================================================
# GOLD TABLE 3 — gold_stage_heatmap
# District × Stage dropout grid for the heatmap chart
# ============================================================
 
# COMMAND ----------
df_sh = spark.table(f"{DB}.silver_dropout_events") \
    .join(df_bhs.select("district_id","bhs_score","national_rank"), on="district_id", how="left") \
    .withColumn("refreshed_at", F.current_timestamp())
 
df_sh.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable(f"{DB}.gold_stage_heatmap")
print(f"✅ gold_stage_heatmap: {df_sh.count()} rows")


In [0]:
spark.sql(f"SHOW TABLES IN {DB}").show()

In [0]:
# GOLD TABLE 4 — gold_survival_curves (from KM results)
# ============================================================
 
# COMMAND ----------
spark.table(f"{DB}.silver_km_curves") \
    .withColumn("refreshed_at", F.current_timestamp()) \
    .write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable(f"{DB}.gold_survival_curves")
print(f"✅ gold_survival_curves written")


In [0]:
# GOLD TABLE 5 — gold_facility_gap
# Specialist availability by district + specialty
# ============================================================
 
# COMMAND ----------
spark.table(f"{DB}.silver_specialty_gaps") \
    .join(df_bhs.select("district_id","district_name","state","bhs_score","national_rank"),
          on="district_id", how="left") \
    .withColumn("refreshed_at", F.current_timestamp()) \
    .write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable(f"{DB}.gold_facility_gap")
print(f"✅ gold_facility_gap written")

In [0]:
# GOLD TABLE 6 — gold_forecast
# Prophet forecast: district-level dropout trend next 90 days
# Run on top-10 worst districts
# ============================================================
 
# COMMAND ----------
top10_ids = [r["district_id"] for r in df_bhs.orderBy("bhs_score", ascending=False).limit(10).collect()]
 
df_ts = spark.table(f"{DB}.silver_patient_journeys") \
    .filter(F.col("district_id").isin(top10_ids)) \
    .groupBy("district_id","district_name","state","report_month") \
    .agg(F.round(F.avg("overall_dropout_rate") * 100, 2).alias("dropout_pct")) \
    .orderBy("district_id","report_month")
 
pdf_ts = df_ts.toPandas()
pdf_ts["report_month"] = pd.to_datetime(pdf_ts["report_month"])
 
forecast_rows = []
for dist_id in top10_ids:
    sub = pdf_ts[pdf_ts["district_id"] == dist_id][["report_month","dropout_pct"]].copy()
    sub.columns = ["ds","y"]
    sub = sub.dropna().sort_values("ds")
    if len(sub) < 12:
        continue
    dist_name = sub["ds"].iloc[0]  # placeholder
    row_info  = df_bhs.filter(F.col("district_id") == dist_id).first()
    dname     = row_info["district_name"] if row_info else str(dist_id)
    dstate    = row_info["state"] if row_info else ""
 
    try:
        model = Prophet(
            yearly_seasonality=True, weekly_seasonality=False,
            daily_seasonality=False, changepoint_prior_scale=0.1
        )
        model.fit(sub)
        future   = model.make_future_dataframe(periods=13, freq="MS")
        forecast = model.predict(future)
        for _, frow in forecast.tail(13).iterrows():
            forecast_rows.append({
                "district_id"  : dist_id,
                "district_name": dname,
                "state"        : dstate,
                "forecast_date": str(frow["ds"].date()),
                "dropout_pct_forecast": round(max(0, min(100, frow["yhat"])), 2),
                "lower_ci"     : round(max(0, frow["yhat_lower"]), 2),
                "upper_ci"     : round(min(100, frow["yhat_upper"]), 2),
                "is_forecast"  : True,
            })
    except Exception as e:
        print(f"  Prophet failed for district {dist_id}: {e}")
 
if forecast_rows:
    df_fc = spark.createDataFrame(pd.DataFrame(forecast_rows)) \
        .withColumn("refreshed_at", F.current_timestamp())
    df_fc.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
        .saveAsTable(f"{DB}.gold_forecast")
    print(f"✅ gold_forecast: {df_fc.count()} forecast rows")
 


In [0]:
# GOLD TABLE 7 — gold_india_summary (KPI cards)
# ============================================================
 
# COMMAND ----------
total_diag    = df_funnel.filter(F.col("scope")=="All India").select("stage0_diagnosed").collect()[0][0]
total_compl   = df_funnel.filter(F.col("scope")=="All India").select("stage6_completed").collect()[0][0]
total_dropped = df_funnel.filter(F.col("scope")=="All India").select("total_dropped").collect()[0][0]
critical_dist = df_bhs.filter(F.col("risk_tier")=="Critical").count()
high_risk_dist= df_bhs.filter(F.col("risk_tier")=="High Risk").count()
worst_stage   = "Stage 2 (PHC → CHC)"  # known from analysis
worst_disease = df_dropout.groupBy("disease_category").agg(
    F.avg("overall_dropout_pct").alias("avg_do")
).orderBy("avg_do", ascending=False).first()["disease_category"]
 
summary_data = [{
    "metric"          : "total_diagnosed",
    "value_num"       : int(total_diag),
    "value_label"     : f"{total_diag/1e6:.2f}M",
    "description"     : "Patients entering system annually",
}, {
    "metric"          : "overall_completion_pct",
    "value_num"       : round(total_compl / max(total_diag, 1) * 100, 1),
    "value_label"     : f"{round(total_compl/max(total_diag,1)*100,1)}%",
    "description"     : "Reach completed treatment",
}, {
    "metric"          : "total_dropped",
    "value_num"       : int(total_dropped),
    "value_label"     : f"{total_dropped/1e6:.2f}M",
    "description"     : "Patients lost in the system annually",
}, {
    "metric"          : "critical_districts",
    "value_num"       : critical_dist,
    "value_label"     : str(critical_dist),
    "description"     : "Districts with BHS > 75 (critical)",
}, {
    "metric"          : "worst_stage",
    "value_num"       : 2,
    "value_label"     : worst_stage,
    "description"     : "Stage with highest dropout volume",
}, {
    "metric"          : "worst_disease",
    "value_num"       : 0,
    "value_label"     : worst_disease,
    "description"     : "Disease with highest dropout rate",
}]
 
spark.createDataFrame(pd.DataFrame(summary_data)) \
    .withColumn("refreshed_at", F.current_timestamp()) \
    .write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable(f"{DB}.gold_india_summary")
print("✅ gold_india_summary written")
 
# ── Final summary ────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("GOLD LAYER COMPLETE")
print("="*60)
for tbl in ["gold_dropout_funnel","gold_bhs_district","gold_stage_heatmap",
            "gold_survival_curves","gold_facility_gap","gold_forecast","gold_india_summary"]:
    cnt = spark.table(f"{DB}.{tbl}").count()
    print(f"  {tbl:<35} {cnt:>8,} rows")
print("="*60)
